# Lab 4: measure your retrieval

Score Lab 3's retriever on 25 labelled questions, add a keyword retriever and fuse the
two, then put an error bar on every score. This notebook follows
[Lab 4 in the book](https://handbook.aggarwal.dev/lab-retrieval-eval.html): read each step there, then run it
here. The code is in `labs/04-retrieval-eval/retrieval_eval.py`. No model is called, so it
costs nothing on any backend.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Before you start: run the tests

13 tests should pass.

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 04-retrieval-eval

## Step 1: labelled questions

Each question names the sections that answer it. Print a direct question and a
paraphrase, and count the kinds.

In [ ]:
sys.path.insert(0, str(root / "labs" / "04-retrieval-eval"))
import retrieval_eval as ev
import rag
from collections import Counter

questions = ev.load_questions()
print(len(questions), "questions:", dict(Counter(q["kind"] for q in questions)))
for q in questions:
    if q["id"] in ("q13", "q14"):
        print(q)

## Step 2: the metrics

Chapter 26's worked example: relevance `0 1 0 1 0` with three relevant passages. Expect
recall@5 = 0.667, a reciprocal rank of 0.5 and nDCG@5 = 0.498.

In [ ]:
ranked = ["a", "b", "c", "d", "e"]
relevant = {"b", "d", "z"}
print("recall@5 ", round(ev.recall_at_k(ranked, relevant, 5), 3))
print("rr       ", ev.reciprocal_rank(ranked, relevant))
print("nDCG@5   ", round(ev.ndcg_at_k(ranked, relevant, 5), 3))

## Step 3: a keyword retriever, BM25

The page's check: chapter 25's three documents, split on whitespace, scored for "fixed
deposit". Expect 0.894, 0.482 and 0.482.

In [ ]:
docs = ["fixed deposit rates for one year", "deposit insurance covers your savings",
        "fixed income funds hold bonds"]
bm = ev.BM25(docs, tokenize=str.split)
print([round(s, 3) for s in bm.scores("fixed deposit")])

## Step 4: fuse the two

Reciprocal rank fusion adds `1 / (60 + rank)` from each list. A passage 1st by vectors
and 3rd by keywords scores 0.03227 and loses to one 2nd and 1st, at 0.03252.

In [ ]:
fused = dict(ev.rrf([["A", "B", "x"], ["B", "y", "A"]]))
print("A (1st, then 3rd):", round(fused["A"], 5))
print("B (2nd, then 1st):", round(fused["B"], 5))

## Step 5: from chunks to sections

Score one question by hand: build the three retrievers at 120 words, take the chunks
for Lab 3's missed question, and turn them into sections. Look at where `loans#loan-periods`
comes in each list.

In [ ]:
chunks = rag.load_chunks(max_words=120)
section_of = {c.id: c.section for c in chunks}
retrievers = ev.build_retrievers(chunks)
question = "How long can I keep a book?"
for name, retriever in retrievers.items():
    ranked = ev.sections(retriever(question), section_of)
    print(f"{name:<6}", ranked[:5])
    print("       rank of loans#loan-periods:", ranked.index("loans#loan-periods") + 1)

## Run it

The whole report, as on the page: the metrics for each retriever at three chunk sizes,
the hit rate by kind of question, and the paired MRR differences. It takes a few seconds.

In [ ]:
!python labs/04-retrieval-eval/retrieval_eval.py

## Step 6: how sure can you be?

`bootstrap_ci` resamples the questions. This reproduces the TF-IDF row at 120 words: an
MRR of 0.77 and an interval of roughly 0.63 to 0.90.

In [ ]:
rows = ev.evaluate(questions, retrievers["tfidf"], section_of)
rr = [r["rr"] for r in rows]
low, high = ev.bootstrap_ci(rr)
print(f"tfidf MRR {ev.mean(rr):.2f}  95% CI [{low:.2f}, {high:.2f}]")

bm25_rr = [r["rr"] for r in ev.evaluate(questions, retrievers["bm25"], section_of)]
diff, lo, hi = ev.paired_difference(bm25_rr, rr)
print(f"bm25 - tfidf: {diff:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]  (contains 0: not shown to differ)")

`--per-question` adds the rank of each question's first relevant section. The questions
every retriever misses are the ones to read first.

In [ ]:
!python labs/04-retrieval-eval/retrieval_eval.py --per-question

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 04-retrieval-eval

## Experiments

Try the [experiments on the lab page](https://handbook.aggarwal.dev/lab-retrieval-eval.html#experiments). A quick
one is to take the heading paths out: run with `--no-headings` and compare hit@1, MRR and
the paraphrase column with the table above.

In [ ]:
!python labs/04-retrieval-eval/retrieval_eval.py --no-headings

Then tick the lab's checklist in the book: your progress is kept in your browser, not in
this notebook.